# Reliability-Aware Adapter -- Experiments

**Goal**: Build an adapter that maps audio features with the overlap embedding to prefix tokens for an LLM

### Import

In [3]:
!pip install mamba-ssm --no-build-isolation

  Using cached mamba_ssm-2.3.1.tar.gz (121 kB)
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=533592144 sha256=d66a3c4c94a693e02d341cace7a6af0b72177b6afa655a25e3a6505130a68cbf
  Stored in directory: /root/.cache/pip/wheels/28/83/54/d45107838fec575b93f5d723f56351cee19a1b13bcd4ec9f3f
Successfully built mamba-ssm


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba
import re
from dataclasses import dataclass

device = torch.device(
    "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu"
    )
)

print(f"Device: {device}")

Device: cuda


### Dimensions & Configs

In [6]:
MODEL_DIM = 1024
LM_DIM = 4096                   # LM hidden dims
AUDIO_DIM = 1024
OVERLAP_FEATURES = 3            # per fram: (is_overlap, n_speakers, overlap_ratio)
OVERLAP_DIM = 32                # output of OverlapEmbedding to learn representation

# Depending on how many frams output from audio encoder
# 1 frame per 20ms = 50 hz
SAMPLE_T = 250
BATCH_SIZE = 4

### Debug ONLY

***Dummy data generators***

These simulate the outputs of frozen feature vectors from encoders.

In [7]:
def fake_audio_features(batch_size: int, T: int) -> torch.Tensor:
    """_summary_

    Args:
        batch_size (int): batch size
        T (int): depending audio length -> T = audio_samples // 320 (16kHz * 20ms = 320 samples per frame)

    Returns:
        torch.Tensors: feature vector from audio encoder -> (batch_size, T, AUDIO_DIM)
    """
    return torch.randn(batch_size, T, AUDIO_DIM)


def fake_overlap(batch_size: int, T: int, overlap_start: float = 0.4, overlap_end: float = 0.7) -> torch.Tensor:
    """_summary_

    Args:
        batch_size (int): batch size 
        T (int): depending audio length -> T = audio_samples // 320 (16kHz * 20ms = 320 samples per frame)
        overlap_start (float, optional): Start of overlap region; fraction of T. Defaults to 0.4.
        overlap_end (float, optional): End of overlap region; fraction of T. Defaults to 0.7.
        
        Assumed Pyannote gives per-frame labels. We convert to 3 features:
        - is_overlap:    0.0 or 1.0 (binary)
        - n_speakers:    1, 2, or 3 (normalized to 0-1 range: 0.33, 0.67, 1.0)
        - overlap_ratio: 0.0 to 1.0 (what fraction of the frame has overlap)
        

    Returns:
        torch.Tensor: convert features per fram into the format that overlapembedding takes -> (batch_size, T, 3)
    """
    feature_vector = torch.zeros(batch_size, T, OVERLAP_FEATURES)

    start_idx = int(T * overlap_start)
    end_idx = int(T * overlap_end)

    # overlap region
    feature_vector[:, start_idx:end_idx, 0] = 1.0       # is_overlap = True
    feature_vector[:, start_idx:end_idx, 1] = 0.67      # n_speaker = 2
    feature_vector[:, start_idx:end_idx, 2] = 0.85      # overlap_ratio = 0.85 (edge case)

    # non-overlap regions
    feature_vector[:, :start_idx, 1] = 0.33
    feature_vector[:, :end_idx, 1] = 0.33

    return feature_vector

In [8]:
# Test
overlap_start = 0.4
overlap_end = 0.7

audio = fake_audio_features(BATCH_SIZE, SAMPLE_T)
overlap = fake_overlap(BATCH_SIZE, SAMPLE_T, overlap_start, overlap_end)
start_region = int(SAMPLE_T * overlap_start)
end_region = int(SAMPLE_T * overlap_end)


print(f"Audio shape: {audio.shape}")
print(f"Overlap shape: {overlap.shape}")
print(f"Overlap region: frames {end_region} - {start_region}  or time {start_region * 0.02}s - {end_region * 0.02}s")
print(f"Overlap values at fram 120: {overlap[0, 120, :]}")
print(f"Clean values at fram 120: {overlap[0, 10, :]}")

Audio shape: torch.Size([4, 250, 1024])
Overlap shape: torch.Size([4, 250, 3])
Overlap region: frames 175 - 100  or time 2.0s - 3.5s
Overlap values at fram 120: tensor([1.0000, 0.3300, 0.8500])
Clean values at fram 120: tensor([0.0000, 0.3300, 0.0000])


### Overlap Embedding Layer (non-linear)

**Purpose**: Convert overlap encoder's raw feature vector into a richer embedding that FiLM can use.

**Flow**: # of raw features -> OVERLAP_DIM (expanded here) -> LM_DIM (FiLM)

**Reasoning**: Expanding from raw features directly to LM_DIM vis FiLM would be a single linear layer, which means it can only learn linear combinations of number of features. The embedding layer adds a nonlinearity (GELU) such that the network can learn meaningful intermediate representations before FiLM uses them.

e.g.

3 -> 4096 (no embedding):

output[347] = $a * overlap + b * speakers + c * ratio + bias$

3 -> 32 (with GELU) -> 4096:

output[347] = $u_0 * e[0] + u_1 * e[1] + ... + u_31 * e[31] + bias$

<br>

**Input**: `(B, T, OVERLAP_FEATURES)`

**Output**: `(B, T, OVERLAP_DIM)`

In [9]:
class OverlapEmbedding(nn.Module):
    def __init__(self, in_features: int = OVERLAP_FEATURES, embed_dim: int = OVERLAP_DIM):
        super().__init__()
        self.embedding = nn.Sequential(
            nn.Linear(in_features, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim)
        )
        
    def forward(self, overlap_info: torch.Tensor) -> torch.Tensor:
        """_summary_

        Args:
            overlap_info (torch.Tensor): feature vector from overlap encoder

        Returns:
            torch.Tensor: learned overlap embeddings -> (B, T, OVERLAP_DIM)
        """
        return self.embedding(overlap_info)

In [10]:
# Test
overlap_embed_layer = OverlapEmbedding()
overlap_embed = overlap_embed_layer(overlap)

print(f"Input: {overlap.shape}")
print(f"output: {overlap_embed.shape}")
print(f"Params: {sum(p.numel() for p in overlap_embed_layer.parameters() if p.requires_grad == True)}")


# Verify: clean vs overlap frames should produce different output
clean_embed = overlap_embed[1, 10, :]
overlap_embed_sample = overlap_embed[1, 100, :]
cos_sim = F.cosine_similarity(clean_embed.unsqueeze(0), overlap_embed_sample.unsqueeze(0))
assert cos_sim < 1.0, "should be < 0 as they take different inputs and shouldn't produce same outputs"
print(f"Cosine similarity (clean vs overlap): {cos_sim.item():.4f}")

Input: torch.Size([4, 250, 3])
output: torch.Size([4, 250, 32])
Params: 1184
Cosine similarity (clean vs overlap): 0.6983


### Conv Compress Block

**Purpose**: Downsample T WavLM frames to T/8 to output each token with a fixed ~160ms window.

***In WaveLM, 1 second = 50 frames, 1 frame = 20 ms (50 Hz)***

**Why Conv and not Q-Former?**

Q-former uses global attention that destroies temportal locality. We can also use window sliced attention while it adds complexity and gains are unknow compared with conv layers.

With current setup, we use downsample 8x to get 160ms window such that the LLM can the localization like "token 15 ≈ time 2.4s". In other words, we tokenize each window and each position corresponds to input frames [8k, 8k+8)

**Architecture**: Conv1d(stride=4) → GELU → Conv1d(stride=2) → GELU = 8x total downsampling

**Input**: `(B, T, 1024)` from WavLM  
**Output**: `(B, T/8, model_dim)` compressed representations

In [20]:
class ConvCompressor(nn.Module):
    def __init__(self, in_dim: int = AUDIO_DIM, out_dim: int = MODEL_DIM):
        super().__init__()
        # We can also use average pooling. It has similar effect as we want here but conv layers give weights to eachi dim.
        self.conv1 = nn.Conv1d(
            in_channels=in_dim,
            out_channels=out_dim,
            kernel_size=4,
            stride=4,
        )
        self.conv2 = nn.Conv1d(
            in_channels=in_dim,
            out_channels=out_dim,
            kernel_size=2,
            stride=2,
        )

        self.gelu = nn.GELU()

    def forward(self, audio_features: torch.Tensor) -> torch.Tensor:
        """_summary_

        Args:
            autio_features (torch.Tensor): output from the audio encoder
        Returns:
            torch.Tensor: compressed feauters -> (B, T//8, out_dim)
        """
        x = audio_features.transpose(1, 2)          #(B, T, 1024) -> (B, 1024, T)
        x = self.conv1(x)                           #(B, 1024, T//4)
        x = self.gelu(x)
        x = self.conv2(x)                           #(B, 1024, T//8)
        x = self.gelu(x)
        x = x.transpose(1, 2)                       #(B, T//8, 1024)
        
        return x

    def get_output_length(self, input_length: int) -> int:
        """_summary_

        Args:
            input_length (int): T

        Returns:
            int: the output T length
        """
        after_conv1 = (input_length - 4) // 4 + 1
        after_conv2 = (after_conv1 - 2) // 2 + 1
        
        return after_conv2

# ---- Test ----
compressor = ConvCompressor().to(device)
compressed = compressor(audio.to(device))

print(f"Input:  {audio.shape}")        # (4, 250, 1024)
print(f"Output: {compressed.shape}")   # (4, 31, 1024)
print(f"Compression: {audio.shape[1]}→{compressed.shape[1]} = {audio.shape[1]/compressed.shape[1]:.1f}x")
print(f"Params: {sum(p.numel() for p in compressor.parameters()):,}")

# Verify the time mapping
N = compressed.shape[1]
print(f"\nTemporal mapping (50Hz input → {N} tokens):")
print(f"  Token 0  → frames [0, 8)    → time 0.00s – 0.16s")
print(f"  Token 15 → frames [120, 128) → time 2.40s – 2.56s")
print(f"  Token {N-1} → frames [{(N-1)*8}, {(N-1)*8+8}) → time {(N-1)*8*0.02:.2f}s – {((N-1)*8+8)*0.02:.2f}s")

# Verify output length calculator matches
for T_test in [250, 500, 100, 750]:
    expected = compressor.get_output_length(T_test)
    actual = compressor(torch.randn(1, T_test, AUDIO_DIM, device=device)).shape[1]
    assert expected == actual, f"Mismatch for T={T_test}: expected {expected}, got {actual}"
print(f"\nOutput length calculator verified for T = [250, 500, 100, 750]")


Input:  torch.Size([4, 250, 1024])
Output: torch.Size([4, 31, 1024])
Compression: 250→31 = 8.1x
Params: 6,293,504

Temporal mapping (50Hz input → 31 tokens):
  Token 0  → frames [0, 8)    → time 0.00s – 0.16s
  Token 15 → frames [120, 128) → time 2.40s – 2.56s
  Token 30 → frames [240, 248) → time 4.80s – 4.96s

Output length calculator verified for T = [250, 500, 100, 750]


### FiLM Conditioning Module 

**Purpose**: Modulate audio features based on overlap detection.

**FiLM** (Feature-wise Linear Modulation): `output = γ(overlap) * audio + β(overlap)`
- `γ` (gamma) = per-dimension scale. During overlap → γ dampens unreliable dims toward 0.
- `β` (beta) = per-dimension shift. During overlap → β pushes representation into a region the LM associates with hedged/uncertain language.

**Comparison with Sigmoid**:

  `Sigmoid gating:   output = [0 to 1] * audio          ← multiply only`

  `FiLM:             output = [anything] * audio + [anything]   ← multiply AND add`

  The + beta is just addition, not an activation. It's the difference between:

  `"I can dim the lights"              ← sigmoid gating`

  `"I can dim the lights AND change the color"   ← FiLM`

In [24]:
class FiLMConditioning(nn.Module):
    def __init__(self, audio_dim: int = AUDIO_DIM, overlap_dim: int = OVERLAP_DIM):
        super().__init__()
        self.gamma = nn.Linear(overlap_dim, audio_dim)
        self.beta = nn.Linear(overlap_dim, audio_dim)

        nn.init.zeros_(self.gamma.weight)
        nn.init.ones_(self.gamma.bias)
        nn.init.zeros_(self.beta.weight)
        nn.init.zeros_(self.beta.bias)
        
    def forward(self, audio: torch.Tensor, overlap_embed: torch.Tensor) -> torch.Tensor:
        gamma = self.gamma(overlap_embed)
        beta = self.beta(overlap_embed)

        return gamma * audio + beta

# ---- Test ----
# Simulate: compressed audio is (B, 31, 4096) and overlap emb needs to match (B, 31, 32)
# In the full model, we'll downsample overlap embeddings to match compressed audio length.
# Here, just test with matching lengths.

test_audio_dim = LM_DIM  # 4096 — the dimension after proj_up in the full model
N_tokens = 31

test_audio = torch.randn(BATCH_SIZE, N_tokens, test_audio_dim, device=device)
test_overlap = torch.randn(BATCH_SIZE, N_tokens, OVERLAP_DIM, device=device)

film = FiLMConditioning(audio_dim=test_audio_dim).to(device)
modulated = film(test_audio, test_overlap)

print(f"Audio input:   {test_audio.shape}")   # (4, 31, 4096)
print(f"Overlap input: {test_overlap.shape}")  # (4, 31, 32)
print(f"Output:        {modulated.shape}")     # (4, 31, 4096)
print(f"Params: {sum(p.numel() for p in film.parameters()):,}")

# Verify identity initialization:
# With weights=0, gamma = 0*overlap + 1 = 1, beta = 0*overlap + 0 = 0
# So output should exactly equal input
diff = (modulated - test_audio).abs().max().item()
print(f"\nIdentity init check: max |output - input| = {diff:.6f}")
print("(Should be ~0.0 — FiLM starts as pass-through)")


Audio input:   torch.Size([4, 31, 4096])
Overlap input: torch.Size([4, 31, 32])
Output:        torch.Size([4, 31, 4096])
Params: 270,336

Identity init check: max |output - input| = 0.000000
(Should be ~0.0 — FiLM starts as pass-through)


### Sequential Context Block

**Purpose**: Give each compressed token context from the entire sequence. Without this, token 5 only knows about frames 40-47 (its conv window). 

#### Mamba

In [ ]:
class MambaContextBlock(nn.Module):
    """Adds sequential context to compressed audio tokens using Mamba SSM.
    
    After conv compression, each token only sees its local 160ms window.
    Mamba scans left-to-right, allowing each token to accumulate information
    from all preceding tokens — so token 20 knows about the clean speech
    at tokens 0-12 AND the overlap starting at token 13.
    
    Uses 1-2 Mamba layers. Each layer:
      - Selective state space: decides what to remember/forget per-step
      - d_state=16: 16-dim hidden state (how much "memory" per step)
      - d_conv=4: local conv within Mamba for fine-grained patterns
      - expand=2: internal expansion factor (2x wider intermediate dim)
    
    """
    def __init__(self, d_model: int = MODEL_DIM, n_layers: int = 1):
        super().__init__()
        
        self.layers = nn.ModuleList([
            Mamba(
                d_model=d_model,
                d_state=16,    # SSM state dimension
                d_conv=4,      # local convolution width
                expand=2,      # internal expansion factor
            )
            for _ in range(n_layers)
        ])
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, N, d_model) — compressed audio tokens
        Returns:
            (B, N, d_model) — contextualized tokens (same shape)
        """
        for layer in self.layers:
            x = x + layer(x)
        return x


# ---- Test ----
mamba_block = MambaContextBlock(d_model=MODEL_DIM, n_layers=1).to(device)
contextualized = mamba_block(compressed.to(device))

print(f"Input:  {compressed.shape}")        # (4, 31, 1024)
print(f"Output: {contextualized.shape}")    # (4, 31, 1024) — same shape
print(f"Params: {sum(p.numel() for p in mamba_block.parameters()):,}")

# Verify residual: output should differ from input (context was added)
diff = (contextualized - compressed).abs().mean().item()
print(f"Mean |output - input|: {diff:.4f} (should be > 0 — context was added)")

Input:  torch.Size([4, 31, 1024])
Output: torch.Size([4, 31, 1024])
Params: 6,666,240
Mean |output - input|: 0.0020 (should be > 0 — context was added)


#### Self-Attention Context Block

We add sinusoidal positional encodding so the model knows the temportal order. Without it, token 15 and token 20 are indistinguishable.

We can also use while it adds complexity that whether the model needs it is unknow.

In [28]:
class SelfAttentionContextBlock(nn.Module):
    def __init__(
        self,
        d_model: int = MODEL_DIM,
        n_head: int = 8,
        n_layers: int = 1,
    ):
        super().__init__()

        # Sinusoidal positional encoding
        # Max 500 tokens covers 4000 frames, 80 seconds, or 80k ms
        self.register_buffer("pos_enc", self._sinusoidal_pe(500, d_model))
        
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_head,
            dim_feedforward=d_model*2,  # match Mamba's expand=2
            batch_first=True,
            norm_first=True,
        )
        self.layers = nn.TransformerEncoder(encoder_layer=self.encoder_layer, num_layers=n_layers)

    @staticmethod
    def _sinusoidal_pe(max_len: int, d_model: int) -> torch.Tensor:
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            -torch.arange(0, d_model, 2) / d_model * torch.log(torch.tensor(10000.0))
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        N = x.shape[1]
        x += self.pos_enc[:, :N, :]
        x = self.layers(x)
        
        return x

# ---- Test ----
attn_block = SelfAttentionContextBlock(d_model=MODEL_DIM, n_layers=1).to(device)
attn_out = attn_block(compressed)

print(f"Input:  {compressed.shape}")      # (4, 31, 1024)
print(f"Output: {attn_out.shape}")        # (4, 31, 1024)
print(f"Params: {sum(p.numel() for p in attn_block.parameters()):,}")
print(f"(Compare to Mamba: {sum(p.numel() for p in mamba_block.parameters()):,} params)")


Input:  torch.Size([4, 31, 1024])
Output: torch.Size([4, 31, 1024])
Params: 16,799,744
(Compare to Mamba: 6,666,240 params)


/tmp/ipykernel_11448/1897881346.py:21: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(encoder_layer=self.encoder_layer, num_layers=n_layers)


### Full Assembled Adapter

Wiring all components together:

```
audio (B,T,1024) ─→ ConvCompressor ─→ ContectEncoder ─→ proj_up ─→ FiLM ─→ MLP ─→ prefix tokens
                                                                     ↑
overlap (B,T,3) ──→ OverlapEmbedding ──→ pool to match length ─────┘
```

**Data flow with shapes (for 5-second clip, B=4)**:
1. `audio (4, 250, 1024)` → ConvCompressor → `(4, 31, 1024)` 
2. `(4, 31, 1024)` → MambaContext → `(4, 31, 1024)`
3. `(4, 31, 1024)` → proj_up Linear → `(4, 31, 4096)`
4. `overlap (4, 250, 3)` → OverlapEmbedding → `(4, 250, 32)` → pool → `(4, 31, 32)`
5. FiLM: `gamma(overlap) * audio + beta(overlap)` → `(4, 31, 4096)`
6. MLP → `(4, 31, 4096)` = **31 prefix tokens for the LLM**


In [40]:
class ReliabilityAwareAdapter(nn.Module):
    def __init__(
        self,
        audio_dim: int = AUDIO_DIM,
        overlap_dim: int = OVERLAP_DIM,
        overlap_features: int = OVERLAP_FEATURES,
        d_model: int = MODEL_DIM,
        lm_dim: int = LM_DIM,
        n_layers: int = 1,
        context_type: str = "mamba",
    ):
        super().__init__()
        self.context_type = context_type

        # ------- Audio Path -------
        self.compressor = ConvCompressor(in_dim=audio_dim, out_dim=d_model)

        # Context Block
        if n_layers < 0:
            raise ValueError(f"n_layers should > 0")
        elif context_type == "mamba" and n_layers > 0:
            self.context = MambaContextBlock(d_model=d_model, n_layers=n_layers)
        elif context_type == "attn" and n_layers > 0:
            self.context = SelfAttentionContextBlock(d_model=d_model, n_layers=n_layers)
        else:
            self.context = nn.Identity()

        self.proj_up = nn.Linear(d_model, lm_dim)

        # ------- Overlap Path -------
        self.overlap_embed = OverlapEmbedding(
            in_features=overlap_features,
            embed_dim=overlap_dim,
        )

        # ------- FiLM Conditioning -------
        self.film = FiLMConditioning(audio_dim=lm_dim, overlap_dim=overlap_dim)

        # ------- MLP -------
        self.mlp = nn.Sequential(
            nn.Linear(lm_dim, lm_dim),
            nn.GELU(),
            nn.Linear(lm_dim, lm_dim),
        )

        
    def forward(
        self,
        audio_features: torch.Tensor,
        overlap_info: torch.Tensor,
    ) -> torch.Tensor:
        # ------- Audio Path -------
        x = self.compressor(audio_features)
        x = self.context(x)
        x = self.proj_up(x)

        N = x.shape[1]

        # ------- Overlap Path -------
        o = self.overlap_embed(overlap_info)            # (B, T, overlap_dim)
        o = o.transpose(1, 2)                           # (B, overlap_dim, T)
        o = F.adaptive_avg_pool1d(o, N)                 
        o = o.transpose(1, 2)                           # (B, N, overlap_dim)

        # ------- MLP -------
        x = self.film(x, o)
        x = self.mlp(x)

        return x

# ---- Test ----
adapter = ReliabilityAwareAdapter(context_type="mamba").to(device)
audio = fake_audio_features(BATCH_SIZE, SAMPLE_T).to(device)
overlap = fake_overlap(BATCH_SIZE, SAMPLE_T).to(device)
prefix_tokens = adapter(audio, overlap)

print(f"Audio input:    {audio.shape}")
print(f"Overlap input:  {overlap.shape}")
print(f"Prefix tokens:  {prefix_tokens.shape}")

total = sum(p.numel() for p in adapter.parameters())
print(f"\nTotal params: {total:,}")

components = {
    "ConvCompressor": adapter.compressor,
    "Context":        adapter.context,
    "proj_up":        adapter.proj_up,
    "OverlapEmbed":   adapter.overlap_embed,
    "FiLM":           adapter.film,
    "MLP":            adapter.mlp,
}
print(f"\nPer-component breakdown:")
for name, module in components.items():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:20s}: {n:>12,} params")


Audio input:    torch.Size([4, 250, 1024])
Overlap input:  torch.Size([4, 250, 3])
Prefix tokens:  torch.Size([4, 31, 4096])

Total params: 50,992,288

Per-component breakdown:
  ConvCompressor      :    6,293,504 params
  Context             :    6,666,240 params
  proj_up             :    4,198,400 params
  OverlapEmbed        :        1,184 params
  FiLM                :      270,336 params
  MLP                 :   33,562,624 params


In [35]:
# ---- Test with different audio lengths ----
# Real recordings vary in duration. Verify the adapter handles this.

print("Variable-length input test:")
for duration_s in [2, 5, 10, 20, 30]:
    T = duration_s * 50  # 50 Hz frame rate
    a = fake_audio_features(1, T).to(device)
    o = fake_overlap(1, T).to(device)
    out = adapter(a, o)
    print(f"  {duration_s:2d}s → T={T:4d} frames → {out.shape[1]:3d} prefix tokens → {out.shape[1]*160/1000:.1f}s coverage")

print("\nAll passed. Adapter handles variable-length inputs correctly.")

Variable-length input test:
   2s → T= 100 frames →  12 prefix tokens → 1.9s coverage
   5s → T= 250 frames →  31 prefix tokens → 5.0s coverage
  10s → T= 500 frames →  62 prefix tokens → 9.9s coverage
  20s → T=1000 frames → 125 prefix tokens → 20.0s coverage
  30s → T=1500 frames → 187 prefix tokens → 29.9s coverage

All passed. Adapter handles variable-length inputs correctly.


## Ablation Variants

Config-driven factory for wandb sweeps. All variants share the same `forward()` interface:
`(B, T, 1024) + (B, T, 3) → (B, N, lm_dim)`

| Variant | Conv | Context | Gating | Purpose |
|---------|------|---------|--------|---------|
| `concat-only` | yes | none | concat overlap → MLP | Is any gating needed? |
| `sigmoid-gate` | yes | none | sigmoid(overlap) * audio | Is FiLM better than simple gating? |
| `film` | yes | none | FiLM | Core novelty, no sequential context |
| `film-attn` | yes | self-attention | FiLM | Does attention beat Mamba? |
| `film-mamba` | yes | 1 Mamba layer | FiLM | **Full model** |
| `film-mamba-2L` | yes | 2 Mamba layers | FiLM | Does more Mamba help? |
| `qformer` | no (Q-Former) | built-in | FiLM | Does temporal locality matter? |

### Concat-only
Overlap info is just concatenated with audio and the MLP deals with it. This is the simplest possible baseline. If this works as well as FiLM, then FiLM isn't needed (unlikely, but you need to prove it).


In [37]:
class ConcatOnlyAdapter(nn.Module):
    """Baseline: concat overlap embeddings with audio, let MLP sort it out."""
    def __init__(self, audio_dim=AUDIO_DIM, overlap_features=OVERLAP_FEATURES,
                 overlap_dim=OVERLAP_DIM, mamba_dim=MODEL_DIM, lm_dim=LM_DIM):
        super().__init__()
        self.compressor = ConvCompressor(in_dim=audio_dim, out_dim=mamba_dim)
        self.overlap_embed = OverlapEmbedding(in_features=overlap_features, embed_dim=overlap_dim)
        self.mlp = nn.Sequential(
            nn.Linear(mamba_dim + overlap_dim, lm_dim),
            nn.GELU(),
            nn.Linear(lm_dim, lm_dim),
        )
    
    def forward(self, audio_features, overlap_info):
        x = self.compressor(audio_features)
        N = x.shape[1]
        o = self.overlap_embed(overlap_info)
        o = o.transpose(1, 2)
        o = F.adaptive_avg_pool1d(o, N)
        o = o.transpose(1, 2)
        x = torch.cat([x, o], dim=-1)
        x = self.mlp(x)
        return x

### Sigmoid Gating
Can only suppress features (0 to 1), cannot shift. Tests whether FiLM's shift (beta) term matters.


In [38]:
class SigmoidGateAdapter(nn.Module):
    """Sigmoid gating: gate = sigmoid(f(overlap)), output = gate * audio."""
    def __init__(self, audio_dim=AUDIO_DIM, overlap_features=OVERLAP_FEATURES,
                 overlap_dim=OVERLAP_DIM, mamba_dim=MODEL_DIM, lm_dim=LM_DIM):
        super().__init__()
        self.compressor = ConvCompressor(in_dim=audio_dim, out_dim=mamba_dim)
        self.overlap_embed = OverlapEmbedding(in_features=overlap_features, embed_dim=overlap_dim)
        self.proj_up = nn.Linear(mamba_dim, lm_dim)
        self.gate = nn.Linear(overlap_dim, lm_dim)
        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, 2.0)  # sigmoid(2) ≈ 0.88
        self.mlp = nn.Sequential(
            nn.Linear(lm_dim, lm_dim),
            nn.GELU(),
            nn.Linear(lm_dim, lm_dim),
        )
    
    def forward(self, audio_features, overlap_info):
        x = self.compressor(audio_features)
        N = x.shape[1]
        x = self.proj_up(x)
        o = self.overlap_embed(overlap_info)
        o = o.transpose(1, 2)
        o = F.adaptive_avg_pool1d(o, N)
        o = o.transpose(1, 2)
        g = torch.sigmoid(self.gate(o))
        x = g * x
        x = self.mlp(x)
        return x

### Q-Former (tests if temporal locality matters)
Queries attend globally — no fixed time-to-token mapping.

In [39]:
class QFormerAdapter(nn.Module):
    """Q-Former baseline: learnable queries cross-attend to all audio frames."""
    def __init__(self, audio_dim=AUDIO_DIM, overlap_features=OVERLAP_FEATURES,
                 overlap_dim=OVERLAP_DIM, lm_dim=LM_DIM, n_queries=32, n_heads=8):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(1, n_queries, audio_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=audio_dim, num_heads=n_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(audio_dim)
        self.overlap_embed = OverlapEmbedding(in_features=overlap_features, embed_dim=overlap_dim)
        self.proj_up = nn.Linear(audio_dim, lm_dim)
        self.film = FiLMConditioning(audio_dim=lm_dim, overlap_dim=overlap_dim)
        self.mlp = nn.Sequential(
            nn.Linear(lm_dim, lm_dim),
            nn.GELU(),
            nn.Linear(lm_dim, lm_dim),
        )
    
    def forward(self, audio_features, overlap_info):
        B = audio_features.shape[0]
        N = self.queries.shape[1]
        q = self.queries.expand(B, -1, -1)
        x, _ = self.cross_attn(q, audio_features, audio_features)
        x = self.norm(x + q)
        x = self.proj_up(x)
        o = self.overlap_embed(overlap_info)
        o = o.transpose(1, 2)
        o = F.adaptive_avg_pool1d(o, N)
        o = o.transpose(1, 2)
        x = self.film(x, o)
        x = self.mlp(x)
        return x

### Factory function: build any variant by name

In [44]:
def build_adapter(variant: str = "film-mamba", **kwargs) -> nn.Module:
    """Build an adapter variant by name. Use this for wandb sweeps.
    
    All variants have the same interface:
        forward(audio_features: (B,T,1024), overlap_info: (B,T,3)) → (B,N,lm_dim)
    """
    variants = {
        "concat-only":   lambda **kw: ConcatOnlyAdapter(**kw),
        "sigmoid-gate":  lambda **kw: SigmoidGateAdapter(**kw),
        "film":          lambda **kw: ReliabilityAwareAdapter(context_type="none", **kw),
        "film-attn":     lambda **kw: ReliabilityAwareAdapter(context_type="attn", n_layers=1, **kw),
        "film-attn-2L":  lambda **kw: ReliabilityAwareAdapter(context_type="attn", n_layers=2, **kw),
        "film-mamba":    lambda **kw: ReliabilityAwareAdapter(context_type="mamba", n_layers=1, **kw),
        "film-mamba-2L": lambda **kw: ReliabilityAwareAdapter(context_type="mamba", n_layers=2, **kw),
        "qformer":       lambda **kw: QFormerAdapter(**kw),
    }
    
    if variant not in variants:
        raise ValueError(f"Unknown variant '{variant}'. Choose from: {list(variants.keys())}")
    
    return variants[variant](**kwargs)


In [48]:
# ---- Test: verify all variants produce the same output shape ----
audio = fake_audio_features(BATCH_SIZE, SAMPLE_T).to(device)
overlap = fake_overlap(BATCH_SIZE, SAMPLE_T).to(device)

print("Variant verification (all should output (4, ~31, 4096)):\n")
for name in ["concat-only", "sigmoid-gate", "film", "film-attn", "film-mamba", "qformer"]:
    try:
        model = build_adapter(name).to(device)
        out = model(audio, overlap)
        params = sum(p.numel() for p in model.parameters())
        print(f"  {name:18s} → {str(tuple(out.shape)):20s}  params: {params:>12,}")
    except Exception as e:
        print(f"  {name:18s} → ERROR: {e}")

Variant verification (all should output (4, ~31, 4096)):

  concat-only        → (4, 31, 4096)         params:   27,405,472
  sigmoid-gate       → (4, 31, 4096)         params:   44,190,880
  film               → (4, 31, 4096)         params:   44,326,048


/tmp/ipykernel_11448/1897881346.py:21: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(encoder_layer=self.encoder_layer, num_layers=n_layers)


  film-attn          → (4, 31, 4096)         params:   61,125,792
  film-mamba         → (4, 31, 4096)         params:   50,992,288
  qformer            → (4, 32, 4096)         params:   42,265,760


### Signal Faithfulness Score (SFS)

**Purpose**: Automatically verify whether the model's generated NL descriptions contain correct numerical claims.

Two components:
1. **ClaimParser**: Extracts `(feature, value, unit)` tuples from generated text via regex
2. **SFSScorer**: Compares parsed claims against SP ground-truth with per-feature tolerances

This is independent of the adapter — you can build and test it now.

In [ ]:
@dataclass
class Claim:
    """A single numerical claim extracted from generated text."""
    feature: str   # e.g., "f0_mean", "snr", "overlap_start"
    value: float   # e.g., 187.0, 28.0, 2.3
    unit: str      # e.g., "Hz", "dB", "s"
    raw_text: str  # the original matched text for debugging


class ClaimParser:
    """Extracts numerical claims from generated speech quality descriptions.
    
    Handles common patterns:
        "F0 = 187 Hz"           → ("f0_mean", 187.0, "Hz")
        "SNR ≈ 28 dB"          → ("snr", 28.0, "dB")
        "SNR of approximately 28 dB" → ("snr", 28.0, "dB")
        "RT60 < 0.15s"         → ("rt60", 0.15, "s")
        "speaking rate: 7 syl/s" → ("speaking_rate", 7.0, "syl/s")
        "overlap at 2.3-4.1s"  → ("overlap_start", 2.3, "s") + ("overlap_end", 4.1, "s")
        "F1 is 542 Hz"         → ("f1_mean", 542.0, "Hz")
    """
    
    # Each pattern: (regex, list of (feature_name, group_index_for_value, unit))
    # Group indices are 1-based.
    PATTERNS = [
        # F0 / pitch
        (r'(?:F0|pitch|fundamental frequency)\s*(?:=|≈|~|is|of)\s*(?:approximately\s+)?(\d+\.?\d*)\s*Hz',
         [("f0_mean", 1, "Hz")]),
        
        # F0 with std dev: "F0 = 187 Hz (σ = 34 Hz)" or "F0 = 187 Hz with a standard deviation of 34 Hz"
        (r'F0\s*=\s*(\d+\.?\d*)\s*Hz\s*\(?σ\s*=\s*(\d+\.?\d*)\s*Hz',
         [("f0_mean", 1, "Hz"), ("f0_std", 2, "Hz")]),
        
        # Formants: F1, F2, F3, F4
        (r'(F[1-4])\s*(?:=|≈|is|of)\s*(?:approximately\s+)?(\d+\.?\d*)\s*Hz',
         [("formant", 2, "Hz")]),  # special handling: feature name includes F1/F2/etc
        
        # SNR
        (r'SNR\s*(?:=|≈|~|is|of)\s*(?:approximately\s+|estimated at\s+)?(\d+\.?\d*)\s*dB',
         [("snr", 1, "dB")]),
        
        # RT60
        (r'RT60\s*(?:=|≈|~|<|>|is)\s*(?:approximately\s+)?(\d+\.?\d*)\s*s',
         [("rt60", 1, "s")]),
        
        # HNR
        (r'HNR\s*(?:=|≈|~|is|of)\s*(?:approximately\s+)?(\d+\.?\d*)\s*dB',
         [("hnr", 1, "dB")]),
        
        # Speaking rate
        (r'(?:speaking rate|rate)\s*(?:=|≈|~|is|of|:)\s*(?:approximately\s+)?(\d+\.?\d*)\s*(?:syl(?:lables)?/s(?:ec(?:ond)?)?)',
         [("speaking_rate", 1, "syl/s")]),
        
        # Spectral tilt
        (r'spectral (?:tilt|slope)\s*(?:=|≈|~|is|of)\s*(?:approximately\s+)?(-?\d+\.?\d*)\s*dB/oct(?:ave)?',
         [("spectral_tilt", 1, "dB/oct")]),
        
        # Jitter
        (r'jitter\s*(?:=|≈|~|is|of)\s*(?:approximately\s+)?(\d+\.?\d*)\s*%',
         [("jitter", 1, "%")]),
        
        # Shimmer
        (r'shimmer\s*(?:=|≈|~|is|of)\s*(?:approximately\s+)?(\d+\.?\d*)\s*%',
         [("shimmer", 1, "%")]),
        
        # VOT
        (r'VOT\s*(?:=|≈|~|is|of)\s*(?:approximately\s+)?(-?\d+\.?\d*)\s*ms',
         [("vot", 1, "ms")]),
        
        # Overlap temporal span: "overlap at 2.3-4.1s" or "overlapping speech from 2.3 to 4.1s"
        (r'overlap(?:ping)?\s*(?:speech\s+)?(?:at|from|during)\s*(\d+\.?\d*)\s*(?:s|sec)?\s*(?:-|to)\s*(\d+\.?\d*)\s*s',
         [("overlap_start", 1, "s"), ("overlap_end", 2, "s")]),
        
        # Sample rate
        (r'(?:sampled at|sample rate)\s*(?:=|≈|~|is|of)?\s*(\d+)\s*(?:kHz|Hz)',
         [("sample_rate", 1, "Hz")]),
    ]
    
    def parse(self, text: str) -> list[Claim]:
        """Extract all numerical claims from generated text.
        
        Args:
            text: generated NL description
        Returns:
            list of Claim objects
        """
        claims = []
        text_lower = text  # keep original case for some patterns
        
        for pattern, extractions in self.PATTERNS:
            for match in re.finditer(pattern, text_lower, re.IGNORECASE):
                for feature, group_idx, unit in extractions:
                    try:
                        value = float(match.group(group_idx))
                        
                        # Special handling for formants: include F1/F2/etc in feature name
                        if feature == "formant":
                            # Find which formant (F1-F4) from the match
                            formant_match = re.search(r'(F[1-4])', match.group(0))
                            if formant_match:
                                feature = f"{formant_match.group(1).lower()}_mean"
                        
                        # Handle kHz → Hz conversion for sample rate
                        if feature == "sample_rate" and "kHz" in match.group(0):
                            value *= 1000
                            unit = "Hz"
                        
                        claims.append(Claim(
                            feature=feature,
                            value=value,
                            unit=unit,
                            raw_text=match.group(0).strip(),
                        ))
                    except (ValueError, IndexError):
                        continue
        
        # Deduplicate: keep first occurrence of each feature
        seen = set()
        unique_claims = []
        for c in claims:
            if c.feature not in seen:
                seen.add(c.feature)
                unique_claims.append(c)
        
        return unique_claims


# ---- Test with the advisor's example output ----
parser = ClaimParser()

example_text = """The recording is sampled at 16 kHz with an estimated SNR of approximately 28 dB.
Segments 0:00-2.3s contain single-speaker speech with F0 = 187 Hz (σ = 34 Hz),
HNR = 18 dB, indicating clear phonation. Speaking rate is 7.0 syl/s with 
syllable-timed rhythm. The average F1 is 542 Hz and F2 is 1483 Hz.
Spectral tilt is -14.3 dB/octave. Segment 2.3-4.1s contains overlapping speech.
VOT for /p/ is 60 ms. Jitter = 0.8%. Shimmer = 1.2%."""

claims = parser.parse(example_text)

print("Parsed claims from example text:\n")
for c in claims:
    print(f"  {c.feature:20s} = {c.value:>8.1f} {c.unit:8s}  ← \"{c.raw_text}\"")

print(f"\nTotal claims extracted: {len(claims)}")

In [ ]:
class SFSScorer:
    """Signal Faithfulness Score — compares parsed claims to SP ground truth.
    
    For each claim, checks if the value is within the tolerance for that feature.
    Computes:
        SFS-Precision: fraction of claims that are correct
        SFS-Recall:    fraction of ground-truth features that were mentioned
        SFS-F1:        harmonic mean
    
    Also provides per-feature breakdown for the paper's analysis table.
    """
    
    # Tolerance thresholds per feature
    # These come from typical within-measurement variability of SP tools
    TOLERANCES = {
        "f0_mean":       5.0,     # ±5 Hz
        "f0_std":        5.0,     # ±5 Hz
        "f1_mean":       30.0,    # ±30 Hz
        "f2_mean":       30.0,    # ±30 Hz
        "f3_mean":       30.0,    # ±30 Hz
        "f4_mean":       30.0,    # ±30 Hz
        "snr":           2.0,     # ±2 dB
        "rt60":          0.05,    # ±0.05 s
        "hnr":           2.0,     # ±2 dB
        "speaking_rate": 0.5,     # ±0.5 syl/s
        "spectral_tilt": 1.5,    # ±1.5 dB/oct
        "jitter":        0.3,     # ±0.3%
        "shimmer":       0.5,     # ±0.5%
        "vot":           8.0,     # ±8 ms
        "sample_rate":   0.0,     # exact match (it's an integer)
    }
    
    # Overlap uses IoU instead of absolute tolerance
    OVERLAP_IOU_THRESHOLD = 0.8
    
    def score(
        self,
        claims: list[Claim],
        ground_truth: dict[str, float],
    ) -> dict:
        """Score claims against ground truth.
        
        Args:
            claims: list of Claim objects from ClaimParser
            ground_truth: dict mapping feature names to true values
                          e.g., {"f0_mean": 189.0, "snr": 27.3, ...}
                          For overlap: {"overlap_start": 2.1, "overlap_end": 4.3}
        Returns:
            dict with "precision", "recall", "f1", and "per_feature" breakdown
        """
        results = []
        
        for claim in claims:
            if claim.feature in ("overlap_start", "overlap_end"):
                # Handle overlap IoU separately
                continue
            
            if claim.feature in ground_truth and claim.feature in self.TOLERANCES:
                gt_value = ground_truth[claim.feature]
                tolerance = self.TOLERANCES[claim.feature]
                error = abs(claim.value - gt_value)
                correct = error <= tolerance
                
                results.append({
                    "feature": claim.feature,
                    "claimed": claim.value,
                    "actual": gt_value,
                    "error": error,
                    "tolerance": tolerance,
                    "correct": correct,
                })
        
        # Handle overlap IoU if both start and end are claimed
        claimed_starts = [c for c in claims if c.feature == "overlap_start"]
        claimed_ends = [c for c in claims if c.feature == "overlap_end"]
        if (claimed_starts and claimed_ends 
                and "overlap_start" in ground_truth and "overlap_end" in ground_truth):
            pred_start = claimed_starts[0].value
            pred_end = claimed_ends[0].value
            gt_start = ground_truth["overlap_start"]
            gt_end = ground_truth["overlap_end"]
            
            # IoU calculation
            inter_start = max(pred_start, gt_start)
            inter_end = min(pred_end, gt_end)
            intersection = max(0, inter_end - inter_start)
            union = (pred_end - pred_start) + (gt_end - gt_start) - intersection
            iou = intersection / union if union > 0 else 0.0
            correct = iou >= self.OVERLAP_IOU_THRESHOLD
            
            results.append({
                "feature": "overlap_span",
                "claimed": f"{pred_start}-{pred_end}s",
                "actual": f"{gt_start}-{gt_end}s",
                "error": 1.0 - iou,
                "tolerance": f"IoU≥{self.OVERLAP_IOU_THRESHOLD}",
                "correct": correct,
            })
        
        # Compute precision, recall, F1
        if results:
            n_correct = sum(1 for r in results if r["correct"])
            precision = n_correct / len(results)
        else:
            precision = 0.0
        
        # Recall: how many ground-truth features were mentioned at all?
        gt_features = set(ground_truth.keys()) - {"overlap_start", "overlap_end"}
        if "overlap_start" in ground_truth:
            gt_features.add("overlap_span")
        
        mentioned = set()
        for r in results:
            mentioned.add(r["feature"])
        
        recall = len(mentioned & gt_features) / len(gt_features) if gt_features else 0.0
        
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        
        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "n_claims": len(results),
            "n_correct": sum(1 for r in results if r["correct"]),
            "n_gt_features": len(gt_features),
            "per_feature": results,
        }


# ---- Test: score the example claims against mock ground truth ----
scorer = SFSScorer()

# Simulate ground truth from SP tools (slightly different from claims — as expected)
ground_truth = {
    "sample_rate":   16000.0,
    "snr":           27.3,     # claimed 28, actual 27.3 → within ±2 dB ✓
    "f0_mean":       189.0,    # claimed 187, actual 189 → within ±5 Hz ✓
    "f0_std":        32.0,     # claimed 34, actual 32 → within ±5 Hz ✓
    "hnr":           18.5,     # claimed 18, actual 18.5 → within ±2 dB ✓
    "speaking_rate": 6.8,      # claimed 7.0, actual 6.8 → within ±0.5 ✓
    "f1_mean":       538.0,    # claimed 542, actual 538 → within ±30 Hz ✓
    "f2_mean":       1490.0,   # claimed 1483, actual 1490 → within ±30 Hz ✓
    "spectral_tilt": -14.0,    # claimed -14.3, actual -14.0 → within ±1.5 ✓
    "vot":           62.0,     # claimed 60, actual 62 → within ±8 ms ✓
    "jitter":        0.9,      # claimed 0.8, actual 0.9 → within ±0.3 ✓
    "shimmer":       1.1,      # claimed 1.2, actual 1.1 → within ±0.5 ✓
    "overlap_start": 2.1,      # claimed 2.3, actual 2.1
    "overlap_end":   4.3,      # claimed 4.1, actual 4.3
}

result = scorer.score(claims, ground_truth)

print("SFS Results:\n")
print(f"  SFS-Precision: {result['precision']:.2f}  ({result['n_correct']}/{result['n_claims']} claims correct)")
print(f"  SFS-Recall:    {result['recall']:.2f}  (features mentioned / {result['n_gt_features']} ground-truth)")
print(f"  SFS-F1:        {result['f1']:.2f}")

print(f"\nPer-feature breakdown:")
for r in result["per_feature"]:
    status = "✓" if r["correct"] else "✗"
    print(f"  {status} {r['feature']:20s}  claimed={str(r['claimed']):>10s}  actual={str(r['actual']):>10s}  err={r['error']}")